<div style="position: relative; width: 100vw; margin-left: calc(-50vw + 50%); left: 50%; transform: translateX(-50%); line-height: 0;">
    <img src="https://raw.githubusercontent.com/pamelaFranco/workshop_glioma/main/Figuras/banner.png"
         style="width: 100vw; max-width: none; height: auto; display: block; border: none;">
    <div style="width: 100vw; height: 8px; background-color: #003366;"></div>
</div>

<div style="height: 180px;"></div>

# **Workshop: Decodificación de Gliomas con ML e IA Interpretable**
# **Proyecto Endowment I+D: DI-07-25/ICS | UNAB - PUC - FALP**
---

# **Bloque 0: Fundamento Científico y Autoría**

In [82]:
from IPython.display import display, HTML

# @markdown

html_intro = """
<div style="background-color: #ffffff; padding: 30px; border-radius: 15px; border: 1px solid #d1d5da; font-family: 'Times New Roman', Times, serif; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
    <div style="text-align: center;">
        <h1 style="color: #1a5276; margin-bottom: 5px; font-variant: small-caps;">Workshop de Decodificación de Gliomas</h1>
        <h3 style="color: #2e86c1; margin-top: 0; font-style: italic;">Basado en Radiómica de T1 y Difusión (DTI)</h3>
    </div>

    <div style="margin: 20px 0; padding: 15px; border-top: 2px solid #1a5276; border-bottom: 2px solid #1a5276; background-color: #f9f9f9;">
        <p style="margin: 0; font-size: 1.1em; line-height: 1.5; text-align: justify;">
            Este laboratorio implementa las características radiómicas descubiertas en el estudio piloto: <br>
            <b>"Beyond Binary Classification: A Pilot Study of Imaging-Derived Glioma Severity Modeling Using T1-Weighted and Diffusion MRI Radiomics"</b>,
            aceptado en <b>Magnetic Resonance Materials in Physics, Biology and Medicine (MAGMA), 2026</b>.
        </p>
    </div>

    <h4 style="color: #1a5276; border-bottom: 1px solid #eee; padding-bottom: 5px;"> Equipo de Investigación</h4>
    <p style="font-size: 1em; color: #333; line-height: 1.4;">
        <b>Autores:</b> Pamela Franco, Cristian Montalba, Raúl Caulier-Cisterna, Ignacio Espinoza, M. Daniela Cornejo, Francisco Torres, Carlos Bennett, Steren Chabert, y Rodrigo Salas.
    </p>

    <h4 style="color: #1a5276; margin-top: 20px;"> Explicación de las Características Seleccionadas (SFS)</h4>
    <p style="font-size: 0.95em; color: #2c3e50;">El estudio identificó 6 biomarcadores clave que explican la severidad del glioma:</p>
    <ul style="font-size: 0.95em; color: #2c3e50; line-height: 1.6;">
        <li><b>Skewness T1:</b> Mide la asimetría de los niveles de gris; indica heterogeneidad en la captación de contraste.</li>
        <li><b>Range T1:</b> La diferencia entre el valor máximo y mínimo de intensidad en el tumor.</li>
        <li><b>GLCM Contrast T1:</b> Mide la variación local de intensidad; a mayor contraste, textura más rugosa y compleja.</li>
        <li><b>GLRLM/GLDM T1:</b> Evalúan la dependencia y longitud de rachas de píxeles, detectando patrones de crecimiento celular.</li>
        <li><b>Robust Mean Absolute Deviation AD (de sus siglas en inglés, axial diffusivity):</b> Indica la dispersión de los datos respecto a la media, eliminando valores atípicos.</li>
    </ul>
</div>
"""
display(HTML(html_intro))

# **Bloque 1: Conexión y Carga de Datos (Automático)**
Este bloque conecta el cuaderno con tu repositorio de GitHub y prepara las variables.

In [ ]:
# @title  1. Conexión con el Repositorio de Datos
import pandas as pd
import numpy as np

URL_RAW = "https://raw.githubusercontent.com/pamelaFranco/workshop_glioma/main/Dataset/dataset_workshop_limpio.csv"

try:
    # utf-8-sig limpia carácteres invisibles al inicio del archivo
    df = pd.read_csv(URL_RAW, encoding='utf-8-sig')
    df.columns = df.columns.str.strip() # Borra espacios en los nombres

    # Definimos columnas
    X = df.drop(['Subjects', 'Classes'], axis=1)
    y = df['Classes']

    print(f"Conexión exitosa. Pacientes totales: {len(df)}")
    display(df.head(3))
except Exception as e:
    print(f"Error: {e}")

Conexión exitosa. Pacientes totales: 36


,Subjects,GLRLM Short Run Low Gray Level Emphasis T1,Range T1,Skewness T1,Robust Mean Absolute Deviation AD,GLCM Contrast T1,GLDM Small Dependence High Gray Level Emphasis T1,Classes
0,Paciente1,0.000992,1823.7720,-1.6990,0.15393,112.2325,1054.9344,2
1,Paciente4,0.000361,2034.3659,-1.2922,0.21727,76.0923,1257.1727,3
2,Paciente5,0.000659,1787.2080,-1.5125,0.13328,81.3991,1063.3296,2


# **Bloque 2: Optimización de Hiperparámetros (Tuning)**


**¿Por qué Random Forest para detectar Gliomas?**

Para este análisis médico, no basta con elegir un algoritmo al azar. Hemos seleccionado **Random Forest (Bosque Aleatorio)** debido a su arquitectura robusta, la cual ofrece ventajas críticas en el diagnóstico clínico:

**1. Estabilidad y Reducción de Error:**
A diferencia de un único árbol de decisión que puede ser "inestable" (cambiar drásticamente con una pequeña variación en los datos), el Random Forest crea cientos de árboles independientes.

* **Voto por Mayoría:** La decisión final no depende de un solo "médico virtual", sino del consenso de todo el bosque. Esto cancela los errores individuales y ofrece un diagnóstico mucho más fiable.

**2. No Interpolación de Valores**
En modelos basados en funciones matemáticas (como regresiones o redes neuronales), el algoritmo intenta "dibujar una curva" entre los puntos. Si un dato es extraño, la curva se deforma.

* **Lógica Basada en Umbrales:** El Random Forest utiliza reglas de decisión puras (ej. ¿Es la expresión del gen X > 0.5?). Esto evita que el modelo "invente" diagnósticos intermedios o resultados que no tienen sentido biológico.

**3. Resistencia al Ruido y Datos Complejos**
Los datos de gliomas suelen ser de "alta dimensión" (muchas variables genéticas).

* **Selección Aleatoria:** Cada árbol solo ve una parte de los datos y una parte de las variables. Esto evita que una sola variable con errores o "ruido" domine y arruine todo el modelo.

In [ ]:
# @title  2. Optimización del Modelo (Hyperparameter Tuning)
# @markdown Este bloque busca la mejor configuración y explica los conceptos técnicos.

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import accuracy_score
from IPython.display import HTML

# 1. Separación de datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Definición del espacio de búsqueda
# Imagina que estos son los "ajustes" de un microscopio para ver mejor las células.
param_dist = {
    'n_estimators': [100, 300, 500],       # ¿Cuántos "médicos" consultamos?
    'max_depth': [None, 5, 10, 20],        # ¿Qué tan compleja es la lógica de cada médico?
    'min_samples_split': [2, 5, 10],       # ¿Cuánta evidencia mínima pide cada médico para decidir?
    'criterion': ['gini', 'entropy']       # ¿Qué fórmula matemática usan para medir el error?
}

print("Buscando la mejor combinación de parámetros...")

# 3. Configuración del buscador
# n_iter=10 -> Probaremos 10 combinaciones de médicos diferentes.
# cv=3 -> Examen en 3 rondas. El modelo debe aprobar 3 exámenes distintos para ser elegido.
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

# Guardamos los resultados
best_model = rf_search.best_estimator_
params = rf_search.best_params_
precisión = accuracy_score(y_test, best_model.predict(X_test))

# --- INTERFAZ EXPLICATIVA DETALLADA ---
html_explainer = f"""
<div style="background-color: #fdfefe; padding: 25px; border-radius: 15px; border: 1px solid #dcdde1; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; color: #2f3640;">
    <h2 style="color: #2980b9; border-bottom: 2px solid #2980b9; padding-bottom: 10px;"> Configuración Optimizada</h2>
    <p>Para lograr una precisión del <b>{precisión:.2%}</b>, la IA ajustó sus componentes así:</p>

    <table style="width:100%; border-collapse: collapse; margin: 20px 0;">
        <tr style="background-color: #f2f2f2;">
            <th style="padding: 12px; text-align: left; border-bottom: 2px solid #ddd;">Parámetro</th>
            <th style="padding: 12px; text-align: left; border-bottom: 2px solid #ddd;">Valor Elegido</th>
            <th style="padding: 12px; text-align: left; border-bottom: 2px solid #ddd;">¿Qué significa realmente?</th>
        </tr>
        <tr>
            <td style="padding: 10px; border-bottom: 1px solid #eee;"><b>n_estimators</b></td>
            <td style="padding: 10px; border-bottom: 1px solid #eee; color: #e67e22;"><b>{params['n_estimators']}</b></td>
            <td style="padding: 10px; border-bottom: 1px solid #eee;">Es el número de árboles. Imagina una junta de <b>{params['n_estimators']} médicos</b>; cada uno da su opinión y la decisión final se toma por mayoría de votos.</td>
        </tr>
        <tr>
            <td style="padding: 10px; border-bottom: 1px solid #eee;"><b>max_depth</b></td>
            <td style="padding: 10px; border-bottom: 1px solid #eee; color: #e67e22;"><b>{params['max_depth']}</b></td>
            <td style="padding: 10px; border-bottom: 1px solid #eee;">Es la "profundidad" del análisis. Si es muy alto, el médico se fija en detalles insignificantes (ruido). Si es bajo, el médico es demasiado simplista.</td>
        </tr>
        <tr>
            <td style="padding: 10px; border-bottom: 1px solid #eee;"><b>min_samples_split</b></td>
            <td style="padding: 10px; border-bottom: 1px solid #eee; color: #e67e22;"><b>{params['min_samples_split']}</b></td>
            <td style="padding: 10px; border-bottom: 1px solid #eee;">Es la prudencia. Indica cuántos casos similares debe ver el modelo antes de crear una nueva regla médica para no inventar patrones donde no los hay.</td>
        </tr>
        <tr>
            <td style="padding: 10px; border-bottom: 1px solid #eee;"><b>CV (Cross-Validation)</b></td>
            <td style="padding: 10px; border-bottom: 1px solid #eee; color: #2980b9;"><b>3 Rondas</b></td>
            <td style="padding: 10px; border-bottom: 1px solid #eee;">Es el control de calidad. El modelo rotó los datos 3 veces para asegurar que su precisión no fue por suerte, sino por aprendizaje real.</td>
        </tr>
    </table>

    <div style="background-color: #27ae60; color: white; padding: 15px; border-radius: 8px; text-align: center; font-size: 1.3em; font-weight: bold;">
         Precisión Final: {precisión:.2%}
    </div>
</div>
"""
display(HTML(html_explainer))

Buscando la mejor combinación de parámetros...


Parámetro,Valor Elegido,¿Qué significa realmente?
n_estimators,300,Es el número de árboles. Imagina una junta de 300 médicos; cada uno da su opinión y la decisión final se toma por mayoría de votos.
max_depth,20,"Es la ""profundidad"" del análisis. Si es muy alto, el médico se fija en detalles insignificantes (ruido). Si es bajo, el médico es demasiado simplista."
min_samples_split,10,Es la prudencia. Indica cuántos casos similares debe ver el modelo antes de crear una nueva regla médica para no inventar patrones donde no los hay.
CV (Cross-Validation),3 Rondas,"Es el control de calidad. El modelo rotó los datos 3 veces para asegurar que su precisión no fue por suerte, sino por aprendizaje real."


# **Bloque 3: Panel Interactivo de Diagnóstico con IA Interpretable (SHAP)**


El mayor problema de los modelos avanzados de IA es que a menudo se comportan como una **"caja negra"**: dan un resultado, pero no explican su razonamiento. En el diagnóstico de Gliomas, esto es inaceptable, ya que las decisiones clínicas deben basarse en evidencia biológica.


Este bloque de código implementa un **Simulador Interactivo** que utiliza la tecnología **SHAP (SHapley Additive exPlanations)** para abrir esa caja negra.

**¿Qué estamos viendo en este panel?**

* **Simulación en Tiempo Real** (`ipywidgets`):

Hemos creado un control deslizante (slider) que te permite navegar por los pacientes del grupo de prueba. El sistema extrae automáticamente el nombre del sujeto, su diagnóstico real y lo compara con la predicción de la IA.

* **Cálculo de Confianza:**

El modelo no solo clasifica, sino que nos dice qué tan seguro está de su decisión (ej. "Confianza: 98.5%"). Esto ayuda al personal médico a identificar casos ambiguos que requieren una segunda revisión humana.


**El Análisis Biofísico: Gráficos de Cascada (Waterfall)**

La pieza central de este panel es el gráfico de SHAP. Su funcionamiento se basa en la teoría de juegos para medir la contribución exacta de cada variable genética o clínica:

* **Punto de Partida ($E[f(X)]$):**
 Es el valor promedio. Si no supiéramos nada del paciente, empezaríamos aquí.

* **Barras Rojas (Aumento de Riesgo):** Son las variables del paciente que "empujan" el diagnóstico hacia el grado detectado. Por ejemplo, una alta expresión de un gen específico.

* **Barras Azules (Disminución de Riesgo):** Son características que protegen al paciente o que alejan el diagnóstico de ese grado en particular.Resultado Final ($f(x)$): Es la suma de todas las fuerzas (rojas y azules). El resultado es el diagnóstico final que vemos en el informe.


**¿Por qué es útil para un médico?**

Este panel permite realizar una **validación clínica**. Si un médico ve que la IA clasificó un Glioma como "Grado 4" porque detectó una mutación que científicamente se sabe que es agresiva, la confianza en el sistema aumenta. Si la IA se basa en datos irrelevantes, el médico puede cuestionar el modelo.

In [ ]:
# @title  3. Panel de Diagnóstico con IA Interpretable
import shap
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, HTML

# 1. Preparar el motor de explicaciones
explainer = shap.TreeExplainer(best_model)
explanation = explainer(X_test)

def simulador_interactivo(id_prueba):
    if id_prueba >= len(X_test): return

    # --- EXTRACCIÓN DE DATOS ---
    idx_real = X_test.index[id_prueba]
    nombre_paciente = df.loc[idx_real, 'Subjects']

    prediccion = best_model.predict(X_test.iloc[[id_prueba]])[0]
    real = y_test.iloc[id_prueba]
    probabilidades = best_model.predict_proba(X_test.iloc[[id_prueba]])[0]
    confianza = np.max(probabilidades) * 100

    # --- LÓGICA DE INTERPRETACIÓN DE CONFIANZA ---
    if confianza >= 85:
        certeza_txt = "ALTA: El modelo identifica patrones muy claros y consistentes."
        emoji_paci = "✅"
        color_confianza = "#27ae60"  # Verde
    elif confianza >= 60:
        certeza_txt = "MEDIA: Existen rasgos mixtos; el caso tiene ambigüedades."
        emoji_paci = "⚠️"
        color_confianza = "#f39c12"  # Naranja
    else:
        certeza_txt = "BAJA: El modelo duda entre varias clases. No tomar como diagnóstico definitivo."
        emoji_paci = "🚨"
        color_confianza = "#e74c3c"  # Rojo

    # --- DISEÑO DEL PANEL DE RESULTADOS (HTML) ---
    # El borde exterior indica si la IA acertó (Verde) o falló (Naranja) comparado con el Grado Real
    color_borde = "#27ae60" if prediccion == real else "#e67e22"

    html_header = f"""
    <div style="border: 3px solid {color_borde}; padding: 25px; border-radius: 15px; background-color: #f8f9fa; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">
        <h2 style="color: #2c3e50; margin-top: 0; border-bottom: 2px solid #ecf0f1; padding-bottom: 10px;">
            Informe de Diagnóstico: {nombre_paciente}
        </h2>

        <table style="width: 100%; margin-top: 15px; border: none;">
            <tr>
                <td style="width: 33%;"><b>Diagnóstico IA:</b><br><span style="font-size: 1.4em; color: #2980b9;">Grado {prediccion}</span></td>
                <td style="width: 33%;"><b>Grado Real (Referencia):</b><br><span style="font-size: 1.4em; color: #7f8c8d;">{real}</span></td>
                <td style="width: 33%;"><b>Nivel de Confianza:</b><br><span style="font-size: 1.4em; color: {color_confianza};">{confianza:.1f}% {emoji_paci}</span></td>
            </tr>
        </table>

        <div style="margin-top: 20px; padding: 15px; background-color: white; border-left: 6px solid {color_confianza}; border-radius: 4px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
            <b style="color: {color_confianza}; text-transform: uppercase; font-size: 0.9em;">Análisis de Certeza:</b><br>
            <span style="color: #34495e; font-size: 1.1em;">{certeza_txt}</span>
        </div>

        <p style="font-size: 0.8em; color: #95a5a6; margin-top: 15px; font-style: italic;">
            * El borde exterior <span style="color: #27ae60; font-weight: bold;">VERDE</span> indica coincidencia con la realidad,
            el <span style="color: #e67e22; font-weight: bold;">NARANJA</span> indica discrepancia.
        </p>
    </div>
    """
    display(HTML(html_header))

    # --- EXPLICACIÓN MÉDICA (SHAP) ---
    print(f"\n ANÁLISIS BIOFÍSICO (¿Por qué la IA seleccionó Grado {prediccion}?):")

    clases = list(best_model.classes_)
    idx_clase = clases.index(prediccion)
    exp_visual = explanation[id_prueba, :, idx_clase]

    plt.figure(figsize=(12, 5))
    shap.plots.waterfall(exp_visual, max_display=10, show=False)
    plt.title(f"Factores que determinan el Diagnóstico de {nombre_paciente}", fontsize=14, pad=20)
    plt.show()

    print("INTERPRETACIÓN DEL GRÁFICO:")
    print("- Los valores en ROJO (derecha) impulsan al modelo a elegir este grado.")
    print("- Los valores en AZUL (izquierda) alejan al modelo de este grado.")

# --- CREACIÓN DEL CONTROL INTERACTIVO ---
slider = widgets.IntSlider(
    min=0,
    max=len(X_test)-1,
    step=1,
    description='Seleccionar Paciente:',
    style={'description_width': 'initial'},
    layout={'width': '600px'},
    continuous_update=False
)

widgets.interact(simulador_interactivo, id_prueba=slider)

interactive(children=(IntSlider(value=0, continuous_update=False, description='Seleccionar Paciente:', layout=…

<function __main__.simulador_interactivo(id_prueba)>

#  **[Importante] Aclaraciones sobre este Código**

> **Nota importante:** El presente cuadernillo es exclusivamente de carácter **educativo y demostrativo**. Presenta limitaciones técnicas que deben considerarse antes de interpretar los resultados en un contexto profesional:

### 1.  Búsqueda Exhaustiva de Hiperparámetros
Los hiperparámetros son como las **"perillas de ajuste"** de una radio. En este código hemos movido algunas para obtener un buen resultado, pero es posible que existan combinaciones más precisas que no hemos explorado debido a la alta demanda de tiempo computacional que esto requiere.

### 2.  Búsqueda de Semilla (Random State)
El algoritmo *Random Forest* construye "bosques" basados en procesos aleatorios. Si cambiamos el número de la semilla (`random_state`), el bosque se construye de forma ligeramente distinta. Un modelo clínico robusto debería demostrar que sus resultados son consistentes sin importar la semilla utilizada, algo que aquí se ha simplificado fijando una única semilla.



### 3.  Nested Cross-Validation (Validación Anidada)
Este es el "estándar de oro" en la validación de modelos, pero requiere más recursos:

* **Validación Normal:** Se divide el dataset en entrenamiento y prueba. Es útil, pero puede haber sesgos.
* **Validación Anidada:** Es un ciclo dentro de otro. El **ciclo interno** busca los mejores parámetros, mientras que el **ciclo externo** evalúa el rendimiento real con datos que el modelo jamás vio durante su ajuste. Sin este proceso, el modelo podría parecer un poco más "inteligente" de lo que realmente es (lo que llamamos *sesgo de selección*).



---
*Este análisis sirve como base para entender el flujo de trabajo de la IA, pero requiere de validación clínica y arquitecturas más complejas para diagnósticos reales.*